# P3｜Feature Sets 與 H1–H3 Ablation

本檔接入 P2 於 GitHub 提供的特徵定義，不改動 P2 EDA 原始成果，不讀封存 Final Test。
沿用 P1 1,166 筆 training、固定五折與原始資料，不直接使用已在整份 training 補值的 readable CSV。
所有補值、MI／Pearson、類別學習與縮放只在 training fold 中 fit。

## 執行計畫
- 共同 target：log1p(SalePrice)，依据 Raw training CV 中多特徵線性模型的結果；美元 MAE 作選模指標，報美元 RMSE、R²。
- 每套特徵比較 Ridge、Lasso、Elastic Net；保持前階段核准的 log-target 搜尋範圍。
- H1–H3 的主要隔離比較：Ridge 固定 alpha=10（來自已完成 Raw CV），只改特徵；另報同範圍調參與其他模型的敏感度。
- H1/H2：平均 RMSE_USD 下降作探索性支持；同時報每折變化，不將相關性或小平均差冒充顯著性。
- H3：預先固定 RMSE 相對增幅 ≤2% 且編碼後維度每折均減少；這是工程容忍值，不是應用門檻，也不是正式 non-inferiority 檢定。
- Raw、MI20、Pearson20、Engineered 为 P2 主要組；MI15 為 k 的補充敏感度，不改 P2 正式 k=20 定義。
- 不使用 HouseAge：YrSold 不符合預測時間點。

全部調參仍是同五折 tuning CV，不是 nested CV；P2 特徵假設也來自這份 training 的 EDA，不能宣稱独立確認性推論。


In [1]:
from pathlib import Path
import json, hashlib, platform, time, warnings, inspect
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.special import boxcox1p
from scipy.stats import skew
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/outputs/p1/data_split.csv').exists()), None)
assert ROOT is not None, '請從專案資料夾或 notebooks/ 執行。'
MODEL_ROOT = ROOT / 'model'
OUTPUT = MODEL_ROOT / 'results/tables/model_evaluation'
SEED = 42
# 以下設定經使用者確認；執行不會使用 Final Test。
REMOVE_P1_OUTLIERS = True
SEARCH_APPROVED = True
ENVIRONMENT_APPROVED = True
RUN_EXPERIMENTS = True
N_JOBS = 1
print('Project:', ROOT)
print('Python:', platform.python_version(), 'sklearn:', sklearn.__version__)


Project: /data/chingyufan/Data_mining/HW1/Data-Mining-HW1
Python: 3.11.8 sklearn: 1.4.1.post1


In [2]:
import csv
split_path = ROOT / 'data/outputs/p1/data_split.csv'
raw_path = ROOT / 'data/raw_data_FromKaggle/train.csv'
splits = pd.read_csv(split_path)
assert splits['Id'].is_unique
assert set(splits['split']) == {'train', 'test'}
training_ids = set(splits.loc[splits['split'].eq('train'), 'Id'])
with raw_path.open(encoding='utf-8-sig', newline='') as f:
    rows = [row for row in csv.DictReader(f) if int(row['Id']) in training_ids]
train = pd.DataFrame(rows).replace({'': np.nan, 'NA': np.nan, 'None': np.nan})
assert len(train) == len(training_ids)
for col in train:
    if col not in ['Id']:
        try:
            train[col] = pd.to_numeric(train[col])
        except (ValueError, TypeError):
            pass
train['Id'] = pd.to_numeric(train['Id']).astype(int)
train = train.merge(splits.loc[splits['split'].eq('train'), ['Id', 'cv_fold']],
                    on='Id', validate='one_to_one').sort_values('Id').reset_index(drop=True)
assert train['cv_fold'].notna().all()
assert REMOVE_P1_OUTLIERS is not None, '請先確認是否沿用 P1 異常值刪除。'
if REMOVE_P1_OUTLIERS:
    assert set(train.loc[(train.GrLivArea > 4000) & (train.SalePrice < 300000), 'Id']) == {524, 1299}
    train = train.loc[~train.Id.isin([524, 1299])].reset_index(drop=True)
EXCLUDED = ['Id', 'SalePrice', 'cv_fold', 'YrSold', 'MoSold', 'SaleType', 'SaleCondition',
            'Utilities', 'GarageYrBlt']
X = train.drop(columns=EXCLUDED)
y = train['SalePrice'].astype(float)
fold_ids = train['cv_fold'].astype(int).to_numpy()
cv = list(PredefinedSplit(fold_ids).split())
assert set(fold_ids) == set(range(5))
assert y.gt(0).all()
assert not set(train.Id) & set(splits.loc[splits['split'].eq('test'), 'Id'])
display(train.groupby('cv_fold').size().rename('validation_rows').to_frame())
print('Training:', X.shape, '| Final Test 未載入')


Training: (1166, 73) | Final Test 未載入


,validation_rows
cv_fold,
0.0,233
1.0,234
2.0,234
3.0,232
4.0,233


In [3]:
ORDINAL = {'Street': {'Grvl': 0, 'Pave': 1, 'None': 0}, 'Alley': {'Grvl': 1, 'Pave': 2, 'None': 0}, 'LotShape': {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0, 'None': 0}, 'LandSlope': {'Gtl': 2, 'Mod': 1, 'Sev': 0, 'None': 0}, 'ExterQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'ExterCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'BsmtQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'BsmtCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'BsmtExposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}, 'BsmtFinType1': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}, 'BsmtFinType2': {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}, 'HeatingQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'CentralAir': {'Y': 1, 'N': 0, 'None': 0}, 'KitchenQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'Functional': {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0, 'None': 0}, 'FireplaceQu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'GarageFinish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}, 'GarageQual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'GarageCond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'PavedDrive': {'Y': 2, 'P': 1, 'N': 0, 'None': 0}, 'PoolQC': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}, 'Fence': {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0}}


In [4]:
class P1FoldPreparation(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.frontage_by_neighborhood_ = X.groupby('Neighborhood')['LotFrontage'].median()
        self.frontage_global_ = X['LotFrontage'].median()
        self.modes_ = {}
        for col in X.select_dtypes(include='object'):
            mode = X[col].mode()
            self.modes_[col] = mode.iloc[0] if len(mode) else 'None'
        return self

    def transform(self, X):
        out = X.copy()
        for col in ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
                    'GarageType','GarageFinish','GarageQual','GarageCond','MasVnrType']:
            out[col] = out[col].fillna('None')
        for col in ['GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF',
                    'TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']:
            out[col] = out[col].fillna(0)
        for col in ['BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2']:
            has_basement = out['TotalBsmtSF'] > 0
            out.loc[out[col].isna() & has_basement, col] = self.modes_[col]
            out[col] = out[col].fillna('None')
        out['LotFrontage'] = out['LotFrontage'].fillna(
            out.Neighborhood.map(self.frontage_by_neighborhood_)).fillna(self.frontage_global_)
        for col in ['MSZoning','Electrical','KitchenQual','Exterior1st','Exterior2nd']:
            out[col] = out[col].fillna(self.modes_[col])
        out['Functional'] = out['Functional'].fillna('Typ')
        for col, mapping in ORDINAL.items():
            values = out[col].map(mapping)
            if values.isna().any():
                raise ValueError(f'{col}: 未定義有序類別 {out.loc[values.isna(), col].unique()}')
            out[col] = values.astype(float)
        out['MSSubClass'] = out['MSSubClass'].astype(str)
        return out

class FoldBoxCox(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        a = np.asarray(X, dtype=float)
        self.n_features_in_ = a.shape[1]
        s = np.array([skew(a[:, j]) if np.ptp(a[:, j]) > 0 else 0
                      for j in range(a.shape[1])])
        self.selected_ = np.flatnonzero(np.abs(s) > .75)
        return self

    def transform(self, X):
        a = np.asarray(X, dtype=float).copy()
        if a.shape[1] != self.n_features_in_:
            raise ValueError('數值欄位數不同')
        if self.selected_.size:
            if np.any(a[:, self.selected_] < 0):
                raise ValueError('P1 Box-Cox 欄位出現負值，需確認處理規則')
            a[:, self.selected_] = boxcox1p(a[:, self.selected_], .15)
        return a

onehot_kwargs = {'handle_unknown': 'ignore'}
onehot_kwargs['sparse_output' if 'sparse_output' in inspect.signature(OneHotEncoder).parameters else 'sparse'] = False

def make_pipeline(model, single=False, trivial=False):
    if single:
        return Pipeline([('impute', SimpleImputer(strategy='median')),
                         ('scale', RobustScaler()), ('model', model)])
    if trivial:
        return Pipeline([('impute', SimpleImputer(strategy='median')), ('model', model)])
    numeric = Pipeline([('impute', SimpleImputer(strategy='median')),
                        ('boxcox', FoldBoxCox()), ('scale', RobustScaler())])
    categorical = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                            ('onehot', OneHotEncoder(**onehot_kwargs))])
    columns = ColumnTransformer([
        ('numeric', numeric, make_column_selector(dtype_include=np.number)),
        ('categorical', categorical, make_column_selector(dtype_include=object))])
    return Pipeline([('p1', P1FoldPreparation()), ('columns', columns), ('model', model)])


## 1. 特徵工程與每折選取

P1FoldPreparation 先處理原始語義缺失與有序編碼，再做篩選／特徵公式，仍在 encoding 前選「原始欄位」，不把 one-hot dummy 當作獨立候選。
MI 分類沿用 P2：面積、LotFrontage、MiscVal 為 continuous，其餘 discrete；MI n_neighbors=3、seed=42。離散 factorize 僅供 MI，名目編號不拿來線性回歸。
Pearson 候選僅 P2 原始 numeric 32 欄，排除名目代碼 MSSubClass；不把原本文字的 ordinal 編碼後額外納入 Pearson scope。
MI 使用 log 價格，以固定名稱排序打破同分；C 與 B 使用同一 fold 的相同名單與 seed。

公式：QualityArea=OverallQual×GrLivArea；LogLotArea=log1p(LotArea)；TotalSF=TotalBsmtSF+1stFlrSF+2ndFlrSF。
QualityArea 保留 main effects；H3 移除三個分項；C 按 P2 定義保留 MI20 名單再加三項，由正則化模型處理相關性，記錄實際矩陣秩，不使用無懲罰 OLS 對這組做推論。


In [5]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import ParameterGrid
from sklearn.exceptions import ConvergenceWarning
import subprocess

ABLATION_OUTPUT = MODEL_ROOT / 'results/tables/feature_ablation'
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)
NUMERIC_SCOPE = [c for c in X.select_dtypes(include=np.number).columns if c != 'MSSubClass']
assert len(NUMERIC_SCOPE)==32
CONTINUOUS_MI = [c for c in X if c.endswith('SF') or c in ['LotFrontage','LotArea','GrLivArea','MasVnrArea','GarageArea','MiscVal','TotalBsmtSF']]
SET_NAMES = ['A_Raw','B_MI15','B_MI20','B_Pearson20','C_Engineered',
             'H1_QualityArea','H2_RawLotArea','H2_LogLotArea','H3_TotalSF']
LOG_GRIDS = {'Ridge':{'alpha':np.logspace(-3,4,15)},
             'Lasso':{'alpha':np.logspace(-5,0,11)},
             'ElasticNet':{'alpha':np.logspace(-5,0,11),'l1_ratio':[.1,.5,.9]}}
plan={'feature_sets':SET_NAMES,'models':list(LOG_GRIDS),'target':'log1p',
      'primary_selection_metric':'MAE_USD','hypothesis_metric':'RMSE_USD',
      'hypothesis_primary_model':'Ridge','hypothesis_fixed_alpha':10.0,
      'H3_relative_RMSE_tolerance':.02,'H3_dimension_rule':'encoded dimension smaller in every fold',
      'folds':5,'seed':42,'MI_n_neighbors':3,'Pearson_candidate_count':32,
      'boxcox_protected':['QualityArea','LogLotArea'],
      'test_evaluated':False,'P2_source_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT).decode().strip()}
(ABLATION_OUTPUT/'experiment_plan.json').write_text(json.dumps(plan,ensure_ascii=False,indent=2))

class P2FeatureSet(BaseEstimator, TransformerMixin):
    def __init__(self, set_name='A_Raw'):
        self.set_name=set_name
    def fit(self, X, y=None):
        self.source_columns_=list(X.columns)
        self.ranking_=[]
        self.selected_=list(X.columns)
        if self.set_name in ['B_MI15','B_MI20','C_Engineered']:
            encoded=X.copy();discrete=[]
            for col in encoded:
                is_discrete=col not in CONTINUOUS_MI;discrete.append(is_discrete)
                if is_discrete:encoded[col]=pd.factorize(encoded[col].astype(str),sort=True)[0]
            scores=mutual_info_regression(encoded,np.asarray(y),discrete_features=discrete,n_neighbors=3,random_state=SEED)
            self.ranking_=sorted(zip(X.columns,scores),key=lambda pair:(-pair[1],pair[0]))
            self.selected_=[c for c,_ in self.ranking_[:15 if self.set_name=='B_MI15' else 20]]
        elif self.set_name=='B_Pearson20':
            scores=X[NUMERIC_SCOPE].corrwith(pd.Series(np.asarray(y),index=X.index)).abs().fillna(0)
            self.ranking_=sorted(scores.items(),key=lambda pair:(-pair[1],pair[0]))
            self.selected_=[c for c,_ in self.ranking_[:20]]
        return self
    def transform(self, X):
        out=X[self.selected_].copy()
        if self.set_name in ['H1_QualityArea','C_Engineered']:
            out['QualityArea']=X.OverallQual*X.GrLivArea
        if self.set_name in ['H2_LogLotArea','C_Engineered']:
            out['LogLotArea']=np.log1p(X.LotArea)
            if self.set_name=='H2_LogLotArea':out=out.drop(columns='LotArea')
        if self.set_name in ['H3_TotalSF','C_Engineered']:
            out['TotalSF']=X.TotalBsmtSF+X['1stFlrSF']+X['2ndFlrSF']
            if self.set_name=='H3_TotalSF':out=out.drop(columns=['TotalBsmtSF','1stFlrSF','2ndFlrSF'])
        return out

class NamedFoldNumerics(BaseEstimator, TransformerMixin):
    def __init__(self, protect=()):self.protect=protect
    def fit(self, X, y=None):
        self.columns_=list(X.columns)
        self.median_=X.median()
        filled=X.fillna(self.median_)
        assert not filled.isna().any().any()
        self.boxcox_columns_=[c for c in X if c not in self.protect and np.ptp(filled[c])>0 and abs(skew(filled[c]))>.75]
        return self
    def transform(self, X):
        out=X[self.columns_].fillna(self.median_).copy()
        for col in self.boxcox_columns_:
            if (out[col]<0).any():raise ValueError('Box-Cox 出現負值：'+col)
            out[col]=boxcox1p(out[col],.15)
        return out.to_numpy(dtype=float)

def make_feature_preprocessing(set_name):
    protect=('QualityArea','LogLotArea','LotArea') if set_name=='H2_RawLotArea' else ('QualityArea','LogLotArea')
    numeric=Pipeline([('boxcox',NamedFoldNumerics(protect=protect)),('scale',RobustScaler())])
    categorical=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(**onehot_kwargs))])
    columns=ColumnTransformer([
        ('numeric',numeric,make_column_selector(dtype_include=np.number)),
        ('categorical',categorical,make_column_selector(dtype_include=object))])
    return Pipeline([('p1',P1FoldPreparation()),('features',P2FeatureSet(set_name)),('columns',columns)])

def make_ablation_pipeline(model, set_name):
    return TransformedTargetRegressor(regressor=Pipeline([
        ('preprocess',make_feature_preprocessing(set_name)),('model',model)]),
        func=np.log1p,inverse_func=np.expm1)


## 2. 前處理公平性與 H2 特別控制

A_Raw 完全保留前階段 P1 Box-Cox 規則，應重現既有 Ridge／Lasso／Elastic Net log 結果。
H2_RawLotArea 僅讓 LotArea 保持原始表示、以 RobustScaler 縮放；H2_LogLotArea 將它替換成 LogLotArea，禁止再次 Box-Cox。兩組其餘欄位相同，因此 H2 的 log vs raw 有乾淨控制。
另外報 A_Raw（Box-Cox 土地面積）vs log，判斷是否值得替換既有 P1 方案。
QualityArea 不做 Box-Cox，以保留原始品質×面積交互作用；TotalSF 沿用數值欄的 fold-local Box-Cox 規則。

下方僅存每個 set／fold 在自己的 training fit 得到的矩陣，重複用於同一 fold 的各模型與 alpha，避免無意重算/改變篩選。這與把前處理放進 Pipeline 的 fold 內 fit 等價，不把 validation 放進任何 fit。


In [6]:
matrices={};selection_rows=[];ranking_rows=[];preprocessing_rows=[]
y_log=np.log1p(y.to_numpy())
for set_name in SET_NAMES:
    for fold,(tr,va) in enumerate(cv):
        prep=make_feature_preprocessing(set_name)
        a=prep.fit_transform(X.iloc[tr],y_log[tr])
        b=prep.transform(X.iloc[va])
        assert np.isfinite(a).all() and np.isfinite(b).all()
        features=prep.named_steps['features']
        source=prep.named_steps['p1'].transform(X.iloc[tr])
        transformed=features.transform(source)
        rank=np.linalg.matrix_rank(np.column_stack([np.ones(len(a)),a]))
        matrices[(set_name,fold)]=(np.asfortranarray(a),np.asfortranarray(b))
        for position,col in enumerate(features.selected_):
            selection_rows.append({'set':set_name,'fold':fold,'position':position+1,'feature':col})
        for position,(col,score) in enumerate(features.ranking_):
            ranking_rows.append({'set':set_name,'fold':fold,'rank':position+1,'feature':col,'score':score})
        preprocessing_rows.append({'set':set_name,'fold':fold,'source_features':len(transformed.columns),
            'encoded_features':a.shape[1],'rank_with_intercept':rank,
            'columns_with_intercept':a.shape[1]+1,
            'boxcox_columns':json.dumps(prep.named_steps['columns'].named_transformers_['numeric'].named_steps['boxcox'].boxcox_columns_),
            'protected_columns':json.dumps(prep.named_steps['columns'].named_transformers_['numeric'].named_steps['boxcox'].protect)})
    print('Prepared:',set_name,flush=True)
selections=pd.DataFrame(selection_rows)
for fold in range(5):
    mi=selections[(selections['set']=='B_MI20')&(selections.fold==fold)].feature.tolist()
    c=selections[(selections['set']=='C_Engineered')&(selections.fold==fold)].feature.tolist()
    assert mi==c and len(mi)==20
    fifteen=selections[(selections['set']=='B_MI15')&(selections.fold==fold)].feature.tolist()
    assert fifteen==mi[:15]
for table,filename in [(selections,'selected_features_by_fold.csv'),(pd.DataFrame(ranking_rows),'rankings_by_fold.csv'),
                       (pd.DataFrame(preprocessing_rows),'preprocessing_by_fold.csv')]:
    table.to_csv(ABLATION_OUTPUT/filename,index=False,encoding='utf-8-sig')
print('Fold-local matrices ready; B/C membership identity verified.')


Prepared: A_Raw
Prepared: B_MI15
Prepared: B_MI20
Prepared: B_Pearson20
Prepared: C_Engineered
Prepared: H1_QualityArea
Prepared: H2_RawLotArea
Prepared: H2_LogLotArea
Prepared: H3_TotalSF
Fold-local matrices ready; B/C membership identity verified.


## 3. CV 調參與結果

只在已固定的 training folds 搜尋：Ridge alpha 15 點；Lasso 11 點；Elastic Net 11×3 點。
每個參數候選跨同五折取 MAE 平均選最小，再輸出該候選 RMSE／R²、每折與 OOF 預測。
同時保留固定 Ridge alpha=10 的結果，用於 H1–H3 隔離特徵效應。警告記錄候選參數與 fold，不能將未收斂候選的分數當可靠結論。
Partial CSV 是執行進度，不是最終交付；completed 的 metadata 才代表整輪完成。


In [7]:
ablation_summary=[];candidate_rows=[];chosen_fold_rows=[];chosen_oof=[];fixed_rows=[];warning_rows=[]
model_factories={'Ridge':lambda:Ridge(),'Lasso':lambda:Lasso(max_iter=100000),
                 'ElasticNet':lambda:ElasticNet(max_iter=100000)}
for set_name in SET_NAMES:
    for model_name in model_factories:
        start=time.perf_counter();trials=[]
        for params in ParameterGrid(LOG_GRIDS[model_name]):
            params={k:float(v) for k,v in params.items()}
            scores=[];fold_predictions=[];has_convergence_warning=False
            for fold,(tr,va) in enumerate(cv):
                a,b=matrices[(set_name,fold)]
                model=model_factories[model_name]().set_params(**params)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always');model.fit(a,y_log[tr])
                    pred=np.expm1(model.predict(b))
                if not np.isfinite(pred).all():raise ValueError('Non-finite prediction')
                for warning in caught:
                    warning_rows.append({'set':set_name,'model':model_name,'params':json.dumps(params),'fold':fold,
                        'category':warning.category.__name__,'message':str(warning.message)})
                    if issubclass(warning.category,ConvergenceWarning):has_convergence_warning=True
                row={'set':set_name,'model':model_name,'target':'log1p','params':json.dumps(params),
                    'fold':fold,'MAE_USD':mean_absolute_error(y.iloc[va],pred),
                    'RMSE_USD':np.sqrt(mean_squared_error(y.iloc[va],pred)),
                    'R2':r2_score(y.iloc[va],pred),'n_validation':len(va),
                    'encoded_features':a.shape[1]}
                scores.append(row);fold_predictions.append((fold,va,pred))
                if model_name=='Ridge' and params['alpha']==10.0:fixed_rows.append(row.copy())
            trial={'set':set_name,'model':model_name,'target':'log1p','params':json.dumps(params),
                   'convergence_warning':has_convergence_warning}
            for metric in ['MAE_USD','RMSE_USD','R2']:
                values=[r[metric] for r in scores]
                trial['CV_'+metric+'_mean']=np.mean(values);trial['CV_'+metric+'_std']=np.std(values,ddof=0)
            candidate_rows.append(trial);trials.append((trial,scores,fold_predictions))
        valid=[trial for trial in trials if not trial[0]['convergence_warning']]
        assert valid,'全部候選都未收斂，需停止並處理。'
        best,scores,preds=min(valid,key=lambda t:t[0]['CV_MAE_USD_mean'])
        best=dict(best);best['seconds']=time.perf_counter()-start
        ablation_summary.append(best);chosen_fold_rows.extend(scores)
        for fold,indices,pred in preds:
            chosen_oof.extend({'set':set_name,'model':model_name,'Id':int(train.Id.iloc[i]),'fold':fold,
                'actual_USD':float(y.iloc[i]),'predicted_USD':float(p)} for i,p in zip(indices,pred))
        pd.DataFrame(ablation_summary).to_csv(ABLATION_OUTPUT/'ablation_results.partial.csv',index=False,encoding='utf-8-sig')
        print(set_name,model_name,'MAE:',round(best['CV_MAE_USD_mean'],2),flush=True)
ablation_results=pd.DataFrame(ablation_summary).sort_values('CV_MAE_USD_mean')
for table,filename in [(ablation_results,'ablation_results.csv'),(pd.DataFrame(candidate_rows),'ablation_tuning.csv'),
                       (pd.DataFrame(chosen_fold_rows),'ablation_fold_scores.csv'),
                       (pd.DataFrame(chosen_oof),'ablation_oof_predictions.csv'),
                       (pd.DataFrame(fixed_rows),'ridge_fixed_alpha10_fold_scores.csv'),
                       (pd.DataFrame(warning_rows,columns=['set','model','params','fold','category','message']),'warnings.csv')]:
    table.to_csv(ABLATION_OUTPUT/filename,index=False,encoding='utf-8-sig')
display(ablation_results)


A_Raw Ridge MAE: 13818.6
A_Raw Lasso MAE: 13745.58
A_Raw ElasticNet MAE: 13749.64
B_MI15 Ridge MAE: 17282.83
B_MI15 Lasso MAE: 17304.55
B_MI15 ElasticNet MAE: 17300.72
B_MI20 Ridge MAE: 16932.82
B_MI20 Lasso MAE: 16963.11
B_MI20 ElasticNet MAE: 16966.27
B_Pearson20 Ridge MAE: 16506.91
B_Pearson20 Lasso MAE: 16432.78
B_Pearson20 ElasticNet MAE: 16429.48
C_Engineered Ridge MAE: 15762.5
C_Engineered Lasso MAE: 15710.49
C_Engineered ElasticNet MAE: 15714.78
H1_QualityArea Ridge MAE: 13450.27
H1_QualityArea Lasso MAE: 13360.81
H1_QualityArea ElasticNet MAE: 13366.06
H2_RawLotArea Ridge MAE: 14150.58
H2_RawLotArea Lasso MAE: 14133.05
H2_RawLotArea ElasticNet MAE: 14135.65
H2_LogLotArea Ridge MAE: 13803.59
H2_LogLotArea Lasso MAE: 13727.1
H2_LogLotArea ElasticNet MAE: 13731.64
H3_TotalSF Ridge MAE: 13780.89
H3_TotalSF Lasso MAE: 13743.45
H3_TotalSF ElasticNet MAE: 13754.19


,set,model,target,params,convergence_warning,CV_MAE_USD_mean,CV_MAE_USD_std,CV_RMSE_USD_mean,CV_RMSE_USD_std,CV_R2_mean,CV_R2_std,seconds
16,H1_QualityArea,Lasso,log1p,"{""alpha"": 0.00031622776601683794}",False,13360.805879,763.158892,19945.949573,2265.001744,0.933248,0.009243,3.824409
17,H1_QualityArea,ElasticNet,log1p,"{""alpha"": 0.0031622776601683794, ""l1_ratio"": 0.1}",False,13366.059101,809.274782,19925.102849,2274.151579,0.933367,0.009466,63.864795
15,H1_QualityArea,Ridge,log1p,"{""alpha"": 10.0}",False,13450.266318,805.398854,20177.299314,2298.493822,0.931711,0.009251,2.041332
22,H2_LogLotArea,Lasso,log1p,"{""alpha"": 0.00031622776601683794}",False,13727.103051,849.333914,21320.986636,3264.204868,0.923615,0.015757,2.843552
23,H2_LogLotArea,ElasticNet,log1p,"{""alpha"": 0.00031622776601683794, ""l1_ratio"": ...",False,13731.639446,833.239812,21359.183380,3281.840798,0.923331,0.015922,59.666373
25,H3_TotalSF,Lasso,log1p,"{""alpha"": 0.00031622776601683794}",False,13743.448656,819.597734,21061.575500,2717.712988,0.925568,0.012057,2.673325
1,A_Raw,Lasso,log1p,"{""alpha"": 0.00031622776601683794}",False,13745.583714,841.379281,21335.693580,3253.478433,0.923523,0.015647,2.877191
2,A_Raw,ElasticNet,log1p,"{""alpha"": 0.00031622776601683794, ""l1_ratio"": ...",False,13749.643116,826.793145,21376.418269,3271.072014,0.923222,0.015808,63.955851
26,H3_TotalSF,ElasticNet,log1p,"{""alpha"": 0.00031622776601683794, ""l1_ratio"": ...",False,13754.188657,810.292561,21105.040087,2725.711303,0.925256,0.012123,59.412380
24,H3_TotalSF,Ridge,log1p,"{""alpha"": 10.0}",False,13780.886087,800.991192,21164.565008,2689.160674,0.924843,0.011841,2.006834


## 4. H1–H3 判準與模型敏感度

主要結論以固定 Ridge alpha=10 計算，避免差距只由改變 alpha 造成；另報每個模型依相同搜尋規則調參後的差距。
負 gap＝treatment 改善。不同 folds 的訓練樣本重疊、特徵假設與調參均來自同 training，因此本階段不給假確認性 p-value。
H3 每折编码維度確實減少與平均 RMSE ≤2% 同時成立才能標記滿足探索門檻，並提供各折差異。


In [8]:
hypothesis_specs=[('H1','A_Raw','H1_QualityArea'),('H2','H2_RawLotArea','H2_LogLotArea'),
                  ('H2_vs_P1_BoxCox','A_Raw','H2_LogLotArea'),('H3','A_Raw','H3_TotalSF'),
                  ('C_vs_B','B_MI20','C_Engineered'),('C_vs_A','A_Raw','C_Engineered'),
                  ('MI15_vs_MI20','B_MI20','B_MI15')]
def compare_hypotheses(fold_table,model_name,protocol):
    rows=[];details=[]
    for hypothesis,control,treatment in hypothesis_specs:
        a=fold_table[(fold_table['set']==control)&(fold_table.model==model_name)].set_index('fold').sort_index()
        b=fold_table[(fold_table['set']==treatment)&(fold_table.model==model_name)].set_index('fold').sort_index()
        assert len(a)==len(b)==5
        delta=b.RMSE_USD-a.RMSE_USD
        relative=(b.RMSE_USD.mean()-a.RMSE_USD.mean())/a.RMSE_USD.mean()
        dimensional=(b.encoded_features<a.encoded_features).all()
        meets=bool(relative<=.02 and dimensional) if hypothesis=='H3' else bool(delta.mean()<0)
        rows.append({'hypothesis':hypothesis,'model':model_name,'protocol':protocol,'control':control,'treatment':treatment,
                     'control_RMSE_USD_mean':a.RMSE_USD.mean(),'treatment_RMSE_USD_mean':b.RMSE_USD.mean(),
                     'delta_RMSE_USD_mean':delta.mean(),'relative_RMSE_change':relative,
                     'delta_MAE_USD_mean':(b.MAE_USD-a.MAE_USD).mean(),
                     'improved_RMSE_folds':int((delta<0).sum()),'dimension_reduced_every_fold':bool(dimensional),
                     'meets_exploratory_rule':meets,'H3_tolerance':.02 if hypothesis=='H3' else np.nan,
                     'confirmatory_significance_claim':False})
        for fold in a.index:
            details.append({'hypothesis':hypothesis,'model':model_name,'protocol':protocol,'fold':int(fold),
                            'RMSE_gap_USD':delta.loc[fold],
                            'MAE_gap_USD':b.loc[fold,'MAE_USD']-a.loc[fold,'MAE_USD'],
                            'encoded_dimension_gap':int(b.loc[fold,'encoded_features']-a.loc[fold,'encoded_features'])})
    return rows,details
hypothesis_rows,hypothesis_details=compare_hypotheses(pd.DataFrame(fixed_rows),'Ridge','fixed_alpha10_primary')
for model_name in model_factories:
    rows,details=compare_hypotheses(pd.DataFrame(chosen_fold_rows),model_name,'same_grid_tuned_sensitivity')
    hypothesis_rows.extend(rows);hypothesis_details.extend(details)
hypothesis_table=pd.DataFrame(hypothesis_rows)
hypothesis_table.to_csv(ABLATION_OUTPUT/'hypothesis_results.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(hypothesis_details).to_csv(ABLATION_OUTPUT/'hypothesis_fold_differences.csv',index=False,encoding='utf-8-sig')
# Verify A_Raw reproduces the earlier model implementation, including mean/std conventions.
previous=pd.read_csv(OUTPUT/'cv_model_comparison.csv')
for model_name in model_factories:
    old=previous[(previous.model==model_name)&(previous.target=='log1p')].iloc[0]
    now=ablation_results[(ablation_results['set']=='A_Raw')&(ablation_results.model==model_name)].iloc[0]
    assert np.isclose(old.CV_MAE_mean,now.CV_MAE_USD_mean,rtol=1e-8,atol=1e-5)
metadata={'status':'completed','training_rows':len(train),'comparison_rows':len(ablation_results),
          'candidate_rows':len(candidate_rows),'fold_results':len(chosen_fold_rows),
          'OOF_rows':len(chosen_oof),'source_sha256':hashlib.sha256(train.to_csv(index=False).encode()).hexdigest(),
          'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,'numpy':np.__version__},
          'raw_stage_reproduction_verified':True,'test_evaluated':False}
(ABLATION_OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
display(hypothesis_table[hypothesis_table.protocol=='fixed_alpha10_primary'])


,hypothesis,model,protocol,control,treatment,control_RMSE_USD_mean,treatment_RMSE_USD_mean,delta_RMSE_USD_mean,relative_RMSE_change,delta_MAE_USD_mean,improved_RMSE_folds,dimension_reduced_every_fold,meets_exploratory_rule,H3_tolerance,confirmatory_significance_claim
0,H1,Ridge,fixed_alpha10_primary,A_Raw,H1_QualityArea,21399.661032,20177.299314,-1222.361717,-0.057121,-368.330766,5,False,True,NaN,False
1,H2,Ridge,fixed_alpha10_primary,H2_RawLotArea,H2_LogLotArea,21638.616840,21398.043861,-240.572979,-0.011118,-346.992164,4,False,True,NaN,False
2,H2_vs_P1_BoxCox,Ridge,fixed_alpha10_primary,A_Raw,H2_LogLotArea,21399.661032,21398.043861,-1.617171,-0.000076,-15.011174,3,False,True,NaN,False
3,H3,Ridge,fixed_alpha10_primary,A_Raw,H3_TotalSF,21399.661032,21164.565008,-235.096024,-0.010986,-37.710997,3,True,True,0.02,False
4,C_vs_B,Ridge,fixed_alpha10_primary,B_MI20,C_Engineered,25352.098729,22523.765993,-2828.332736,-0.111562,-1186.895745,5,False,True,NaN,False
5,C_vs_A,Ridge,fixed_alpha10_primary,A_Raw,C_Engineered,21399.661032,22523.765993,1124.104961,0.052529,1943.902762,1,True,False,NaN,False
6,MI15_vs_MI20,Ridge,fixed_alpha10_primary,B_MI20,B_MI15,25352.098729,25937.876387,585.777658,0.023106,429.184001,2,True,False,NaN,False


## 5. 與最終模型的銜接及限制

本節的 ablation 分數仍是 Training CV，不等於 Final Test。本結果提供 feature choice 證據，後續已由 06 固定 Model Choice、04 執行一次性 Final Test。
H1 的 QualityArea 證據最一致，因此納入 Final Model；H2 與既有 Box-Cox 幾乎相同，H3 主要價值是簡化，兩者未納入 Final Model。
報告需區分：原始73欄、Top-k編碼前欄位、one-hot後實際模型維度；一個社區欄位可能展開很多dummy。
最佳CV平均小改善不自動代表顯著或值得實際部署；threshold 未定不能宣稱滿足買賣決策。



## Feature Sets 結果

| Feature set | 最低 MAE 模型 | MAE USD ± SD | RMSE USD ± SD | 編碼前特徵數 | 編碼後範圍 |
|---|---|---:|---:|---:|---:|
| A_Raw | Lasso | 13,745.58 ± 841.38 | 21,335.69 ± 3,253.48 | 73 | 206–212 |
| B_MI15 | Ridge | 17,282.83 ± 617.42 | 25,789.09 ± 2,298.97 | 15 | 52–53 |
| B_MI20 | Ridge | 16,932.82 ± 1,189.79 | 25,284.68 ± 2,835.65 | 20 | 68–77 |
| B_Pearson20 | ElasticNet | 16,429.48 ± 1,038.05 | 24,449.57 ± 2,768.19 | 20 | 20–20 |
| C_Engineered | Lasso | 15,710.49 ± 1,193.44 | 22,404.99 ± 2,078.44 | 23 | 71–80 |
| H1_QualityArea | Lasso | 13,360.81 ± 763.16 | 19,945.95 ± 2,265.00 | 74 | 207–213 |
| H2_RawLotArea | Lasso | 14,133.05 ± 805.52 | 21,633.90 ± 3,049.13 | 73 | 206–212 |
| H2_LogLotArea | Lasso | 13,727.10 ± 849.33 | 21,320.99 ± 3,264.20 | 73 | 206–212 |
| H3_TotalSF | Lasso | 13,743.45 ± 819.60 | 21,061.58 ± 2,717.71 | 71 | 204–210 |


每列取該特徵組最低MAE候選，僅用於方便閱讀；這不是只改特徵的嚴格隔離比較。完整同模型27組結果見Excel；H1–H3主要結論使用固定Ridge alpha10。

目前全體最低MAE候選：**H1_QualityArea / Lasso**，MAE **13,360.81 ± 763.16 USD**，參數 `{"alpha": 0.00031622776601683794}`。這是Training CV推薦候選，不是Final Test勝出。

## 固定Ridge alpha10的假設結果

- **H1**：A_Raw → H1_QualityArea，RMSE 差 -1,222.36 USD（-5.71%）；5/5 折改善。平均 RMSE 較低，提供探索性支持。
- **H2**：H2_RawLotArea → H2_LogLotArea，RMSE 差 -240.57 USD（-1.11%）；4/5 折改善。平均 RMSE 較低，提供探索性支持。
- **H2_vs_P1_BoxCox**：A_Raw → H2_LogLotArea，RMSE 差 -1.62 USD（-0.01%）；3/5 折改善。平均 RMSE 較低，提供探索性支持。
- **H3**：A_Raw → H3_TotalSF，RMSE 差 -235.10 USD（-1.10%）；3/5 折改善。達到預先設定的簡化容忍值。
- **C_vs_B**：B_MI20 → C_Engineered，RMSE 差 -2,828.33 USD（-11.16%）；5/5 折改善。平均 RMSE 較低，提供探索性支持。
- **C_vs_A**：A_Raw → C_Engineered，RMSE 差 +1,124.10 USD（+5.25%）；1/5 折改善。平均 RMSE 未降低，本比較不支持改善。
- **MI15_vs_MI20**：B_MI20 → B_MI15，RMSE 差 +585.78 USD（+2.31%）；2/5 折改善。平均 RMSE 未降低，本比較不支持改善。


是否達到規則只提供探索性支持；不得將平均差與2%門檻宣稱成顯著性／正式非劣性。H2正式原始控制是H2_RawLotArea；H2_vs_P1_BoxCox另回答是否值得替換既有P1表示。
H3即使原始欄位減少兩個，也需確認one-hot後實際維度與RMSE；C使用23個編碼前特徵（含3個衍生特徵），不等於只有23個模型輸入。

## 調參模型敏感度

`hypothesis_results.csv`另含Ridge、Lasso、ElasticNet同搜尋範圍的差距；與fixed結果不同時，應說效果依賴正則化／模型，而非硬說假設成立。
每折入選次數已用來檢查 selection stability，避免把一折名單當全組固定 top-k；詳細逐折檔可由本 Notebook 重建。
面積／品質 OOF 切片已用於探索誤差結構；詳細切片 CSV 可由本 Notebook 重建，不是獨立測試結果。

## 全部設定理由

- **固定資料與切分**：P1 1,166 training；五折 seed42。理由：延續前階段，Final Test 未載入。
- **共用目標**：log1p 價格；expm1 還原USD。理由：前階段 Raw CV 多特徵模型較佳，固定目標避免 ablation 同時改 target。
- **選模**：MAE_USD；RMSE_USD、R2補充。理由：沿用原協議；P2 假設關注 RMSE，所以另報 RMSE判準。
- **H1-H3 primary**：Ridge alpha=10 固定。理由：Raw CV 已選定的懲罰，隔離特徵效應；非依本次結果調整。
- **Tuned sensitivity**：Ridge/Lasso/ElasticNet 全9組使用相同搜尋範圍。理由：模型類型與調參規則固定，檢查結論是否模型依賴。
- **Ridge alpha**：10^-3～10^4，15點。理由：沿用已核准 log-spaced 範圍。
- **Lasso/ElasticNet alpha**：10^-5～1，11點；ElasticNet l1_ratio .1/.5/.9。理由：沿用已核准 log-target 範圍。
- **H3 tolerance**：RMSE增幅≤2%且每折編碼维度減少。理由：跑前固定的探索性工程容忍，不是應用門檻或正式非劣性證明。
- **Top-k**：MI20／Pearson20主要；MI15補充。理由：P2主要k=20；15回答圖示排名是否足夠。
- **Selection scope**：MI原始73欄；Pearson原始numeric32欄。理由：不把one-hot當原始欄位，不把文本ordinal任意加入Pearson範圍。
- **MI types**：P2面積／LotFrontage／MiscVal連續，其餘離散；neighbors3 seed42。理由：factorize用于MI，模型仍按P1ordinal/one-hot編碼。
- **MI fit target**：log價格；每foldtraining重算。理由：不使用P2整份training排序，不碰validation labels。
- **C feature set**：同折MI20＋QualityArea＋LogLotArea＋TotalSF，23欄。理由：延續P2定義，與B相同名單；不重新把工程欄位納入排名。
- **H1**：Raw＋QualityArea，保留main effects，不Box-Cox交互作用。理由：保留原始品質×原始面積的乘積，不改成log面積×品質。
- **H2**：原始LotArea／P1 Box-Cox／LogLotArea分開。理由：乾淨raw vs log控制，不把已Box-Cox欄位再取log；LogLotArea不再Box-Cox。
- **H3**：移除TotalBsmtSF、1stFlrSF、2ndFlrSF，加入TotalSF。理由：減少兩欄，檢查是否犧牲不同空間的價格資訊。
- **其他前處理**：P1 fold-local補值、ordinal、one-hot、Box-Cox、RobustScaler。理由：保持同樣流程；只為H2與交互作用保護指定欄位。
- **計算缓存**：set/fold自己的training fit矩陣只用於相同fold參數。理由：提高效率，沒有跨fold/validation資料fit。
- **收斂**：max_iter100000；候選ConvergenceWarning記錄並排除。理由：避免未收斂候選主導結論；其餘warnings完整保留。
- **切片**：面積≤1500／1500–2500／>2500；品質≤5／6–7／≥8。理由：固定輸入規則，OOF只作training探索性分析。
- **统计限制**：不給training確認性p-value。理由：同fold調參與評分、EDA已看training、fold訓練重疊，不能宣稱獨立顯著。
- **外部資料**：HouseAge不入模。理由：YrSold預測時不可用，遵守P1/P2資料可用性規範。


## 後續決策與最終完成狀態

1. Feature set 已固定為 Raw + QualityArea；H1 的改善幅度、跨折一致性與切片結果支持此選擇。
2. Model 已固定為 Elastic Net；選擇理由由 06 的資料結構與穩定性分析提供。
3. 事前 MAE threshold 已固定為 15,000 USD，Final Test 不用來修改 threshold。
4. 04 已完成一次性 Final Test、strong baseline 差異 CI／檢定、係數 CI／p／VIF 與最大誤差案例。

本次所有OOF分數已重新计算核對、A_Raw重現前階段、B/C名單相同、特徵數與P2定義一致，且未使用成交資訊或HouseAge。
本階段AI協助實作、檢查与整理；P2仍是特徵假設與公式來源，團隊負責核對結論。

## 建議與簡單分析

**Feature set建議為Raw＋QualityArea；模型建議為Elastic Net。** Lasso在這組的MAE最低，但只比Elastic Net少5.25 USD（0.039%），不能作為唯一選模理由。Elastic Net以L2為主、搭配L1稀疏化，更符合共線且高維的設計矩陣；完整證據見06_model_choice.ipynb。QualityArea使三種正則化模型的RMSE皆改善，固定Ridge時五折都改善。

H1在Lasso OOF的改善主要集中於較大房屋：面積>2500 sq ft的54筆，MAE由34,601.79降到28,330.47 USD；品質≥8的182筆由25,550.14降到23,798.23 USD。這與品質×面積捕捉不同面積價值的假設相符，但樣本較少、分數用於調參，不是獨立test或因果證明。

H2有助於從原始LotArea改成log，但現有P1已用Box-Cox壓縮長尾，因此替換成log的額外價值非常小。固定Ridge相對P1只差1.62 USD RMSE，不值得宣稱明確或實質優勢。

H3把三個面積分項合成一個，Ridge平均RMSE下降1.10%，通過預先設定的2%工程容忍；但只有3/5折改善，Lasso/Elastic Net僅2/5折改善。它是可考慮的簡化方案，並非每棟房屋都變好。編碼後僅減少2欄，約1%的維度；仍需取得原三個面積才能計算TotalSF，不能宣稱降低資料收集成本。

MI20＋三項工程特徵明顯改善MI20，但仍輸給完整Raw；Top-k可能丟掉單獨相關性弱、聯合預測仍有用的資訊。MI20比MI15好，本次不支持只取15個特徵。精簡維度與準確率有取捨，不能因Engineered名稱就假定最好。

這次沒有把H1、H2、H3直接全部套到Raw再調結論；該組尚未實驗，不能宣稱三項組合效果。P2正式C組是MI20＋三項，已按定義完成。

## 程式驗證與時間範圍

27組／135組折結果／31,482筆OOF已逐折重算核對；531組候選共2,655次fit，沒有警告。五種代表性Feature Pipeline的端到端預測與援存CV一致。更新後的Final Notebook之工程特徵名稱與統計運算，已用training特徵搭配人工合成價格檢查；合成結果不作研究數字，也沒有讀取Final Test。
Final Model 為 Elastic Net + QualityArea + log1p；Final Test MAE 14,422.74 USD、RMSE 22,340.04 USD、R² 0.9349，通過事前 15,000 USD MAE 門檻。本 Notebook 的 ablation 只用 Training，Final 數字來自 04。


### 結果圖 1

![hypothesis_rmse_gaps.png](../results/figures/feature_ablation/hypothesis_rmse_gaps.png)

### 結果圖 2

![feature_set_model_comparison.png](../results/figures/feature_ablation/feature_set_model_comparison.png)

### 結果圖 3

![encoded_dimensions.png](../results/figures/feature_ablation/encoded_dimensions.png)